In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.odr import ODR, Model, RealData
from math import floor, log10


def analysis(excel_file, sheet_name, y_error, t_exp_error, ball):

  # creating the data table
  df = pd.read_excel(excel_file, sheet_name=sheet_name)
  df["t_avg"] = df[["t1 (s)", "t2 (s)", "t3 (s)"]].mean(axis=1)
  df["t2"] = df["t_avg"] ** 2
  df["y_error"] = y_error
  df["t_exp_error"] = t_exp_error
  df["t_st_error"] = df[["t1 (s)", "t2 (s)", "t3 (s)"]].std(axis=1)
  df["t_total_error"] = (df["t_exp_error"]**2 + df["t_st_error"]**2)**0.5
  df["t2_error"] = 2*df["t_avg"]*df["t_total_error"]

  x = df["t2"]
  y = df["y (m)"]

  xerr = df["t2_error"]
  yerr = df["y_error"]

  # Linear model for ODR
  def linear(beta, x):
    return beta[0] * x + beta[1]

  # Create model and data objects
  model = Model(linear)
  data = RealData(x, y, sx=xerr, sy=yerr)

  # Initial guesses for slope and intercept
  odr = ODR(data, model, beta0=[1.0, 0.0])

  # Run fit
  output = odr.run()

  # Best-fit parameters
  m, b = output.beta

  # Parameter uncertainties
  m_err, b_err = output.sd_beta

  # Generate fit line
  x_fit = np.linspace(min(x)-0.02, max(x)+0.02, 100)
  y_fit = linear([m, b], x_fit)

  # Plot
  plt.figure(figsize=(8,6))

  plt.errorbar(
  x,
  y,
  xerr=xerr,
  yerr=yerr,
  fmt='o',
  capsize=4,
  color='blue',
  label='Experimental Data'
  )

  plt.plot(
  x_fit,
  y_fit,
  'r-',
  linewidth=2,
  label=(
  f'm = {m:.3f} ± {m_err:.3f}\n'
  f'b = {b:.3f} ± {b_err:.3f}'
  )
  )

  plt.xlabel(r'$t^2$ (s$^2$)', fontsize=14)
  plt.ylabel('Distance (m)', fontsize=14)
  plt.title(f'Distance vs Time Squared - {ball}')
  plt.grid(True)
  plt.legend()
  plt.tight_layout()

  plt.savefig(f"free_fall_fit_{sheet_name}.png", dpi=300, bbox_inches="tight")

  plt.show()

  print(f"Analysis for {ball}:")

  g = 2 * m
  g_err = 2 * m_err

  # Determine decimal place corresponding to 1 significant digit in the error
  decimals = -floor(log10(abs(g_err)))

  # Round error to 1 significant digit
  g_err_rounded = round(g_err, decimals)

  # Round value to same decimal place
  g_rounded = round(g, decimals)

  print(f"g = {g_rounded:.{decimals}f} ± {g_err_rounded:.{decimals}f} m/s²")

  g_accepted = 9.81
  percent_difference = abs(g - g_accepted) / g_accepted * 100

  print(f"Percentage difference = {percent_difference:.2f}%")

  return df


y_error = 0.001
t_exp_error = 0.01
excel_file = "free_fall_Data_Table.xlsx"
sheet_name_1 = "ball_15.64_mm"
sheet_name_2 = "ball_19.00_mm"
ball_1 = "15.64 mm ball"
ball_2 = "19.00 mm ball"

df1 = analysis(excel_file, sheet_name_1, y_error, t_exp_error, ball_1)
df2 = analysis(excel_file, sheet_name_2, y_error, t_exp_error, ball_2)

